In [28]:
import pandas as pd
data = pd.read_csv("irrigation_dataset.csv")
print(data.head())

   temperature  humidity  rainfall  soil_type  crop_stage
0           25        84       175          3           1
1           35        42       200          3           1
2           20        43       151          1           3
3           39        87        27          3           3
4           40        44        13          2           1


In [30]:
import numpy as np

def generate_label(row):
    score = 0

    if row['rainfall'] < 50:
        score += 1
    if row['humidity'] < 60:
        score += 1
    if row['temperature'] > 32:
        score += 1
    if row['soil_type'] in [2, 3]:
        score += 1
    noise = np.random.choice([0,1], p=[0.85,0.15])

    if score >= 2:
        return 1 if noise == 0 else 0
    else:
        return 0 if noise == 0 else 1

In [31]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score
import pickle

# Load dataset
data = pd.read_csv("irrigation_dataset_with_labels.csv")

X = data[["temperature", "humidity", "rainfall", "soil_type", "crop_stage"]]
y = data["irrigation"]

# Flip some predictions manually
mask = np.random.rand(len(y)) < 0.1
y[mask] = 1 - y[mask]
# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Model
model = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(kernel='rbf', C=10, gamma=0.1))
])

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

#  Metrics
accuracy = accuracy_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("MSE:", mse)
print("RMSE:", rmse)
print("R2 :", r2)

# Save model
pickle.dump(model, open("irrigation_model.pkl", "wb"))

Accuracy: 0.8666666666666667
MSE: 0.13333333333333333
RMSE: 0.3651483716701107
R2 : 0.31818181818181845
